In [6]:
import numpy as np
import pandas as pd

# ==========================================
# 1. PARÂMETROS FINANCEIROS DE ENTRADA
# ==========================================
t_lifespan = 20           
r_discount = 0.065         
fuel_euro_per_L = 0.60    
carbon_euro_per_kgCO2 = 86.65/1e3    
rho = 0.95 
EFHFO = 3.15    
LHV = 11.25e3    
efficency_thermal = 0.36   
fuel_euro_per_kgHFO = fuel_euro_per_L / rho      

fuel_eWh_per_kgHFO = LHV * efficency_thermal #eletric energy per kg of HFO
fuel_euro_per_Wh = fuel_euro_per_kgHFO/fuel_eWh_per_kgHFO

fuel_eWh_per_kgCO2 = fuel_eWh_per_kgHFO / EFHFO
carbon_euro_per_Wh = carbon_euro_per_kgCO2 / fuel_eWh_per_kgCO2 # €/Wh


# Custos unitários da tecnologia (Ajusta com os custos do teu enunciado/grupo se diferirem)
capex_solar_per_MW = 900000.0       # €/MW
capex_wind_per_MW = 1400000.0       # €/MW
capex_hydro_per_MW = 2900000.0      # €/MW
capex_thermal_per_MW = 1200000.0     # €/MW

opex_solar_per_MW = 0.018 * capex_solar_per_MW         # €/MW/ano
opex_wind_per_MW = 0.025 * capex_wind_per_MW         # €/MW/ano
opex_hydro_per_MW = 0.03 * capex_hydro_per_MW        # €/MW/ano
opex_thermal_fixed_per_MW = 0.04 * capex_thermal_per_MW # €/MW/ano



# ==========================================
# 2. CAPACIDADES INSTALADAS E ENERGIA ANUAL
# ==========================================
# (Usa as variáveis já declaradas no teu notebook)
P_solar_MW = 10
P_wind_MW = 9.4
P_hydro_MW = 0.84
P_thermal_MW = 7.23
E_thermal_annual_MWh = 24241

# ==========================================
# 3. TABELA 14: CÁLCULO DE CAPEX E OPEX
# ==========================================
capex_vre_hydro = (P_solar_MW * capex_solar_per_MW + 
                   P_wind_MW * capex_wind_per_MW + 
                   P_hydro_MW * capex_hydro_per_MW)

capex_thermal = P_thermal_MW * capex_thermal_per_MW

first_year_opex = (P_solar_MW * opex_solar_per_MW +
                   P_wind_MW * opex_wind_per_MW +
                   P_hydro_MW * opex_hydro_per_MW +
                   P_thermal_MW * opex_thermal_fixed_per_MW)


final_year_opex_yr20 = first_year_opex * ((1 + r_discount) ** (t_lifespan - 1))

table_14_df = pd.DataFrame({
    'Metric': ['Total VRE & Hydro CAPEX (€)', 'Thermal Backup CAPEX (€)', 'First-Year OPEX (€/year)', 'Final-Year OPEX (Year 20) (€/year)'],
    'Value': [capex_vre_hydro, capex_thermal, first_year_opex, final_year_opex_yr20]
})

print("--- TABELA 14: CAPEX & OPEX Assumptions ---")
display(table_14_df.round(2))



--- TABELA 14: CAPEX & OPEX Assumptions ---


,Metric,Value
0,Total VRE & Hydro CAPEX (€),24596000.00
1,Thermal Backup CAPEX (€),8676000.00
2,First-Year OPEX (€/year),911120.00
3,Final-Year OPEX (Year 20) (€/year),3014519.71


In [7]:
total_capex = capex_vre_hydro + capex_thermal
total_fixed_opex = first_year_opex
variable_opex = E_thermal_annual_MWh * (fuel_euro_per_Wh + carbon_euro_per_Wh) * 1e6
total_opex = total_fixed_opex + variable_opex
solar_energy_annual_MWh = 20306
wind_energy_annual_MWh = 37321
hydro_energy_annual_MWh = 4578
solar_d = 0.005
wind_d = 0.01
hydro_d = 0.02
thermal_d = 0.03


# Vector de anos (1 a 20) e fatores de desconto NPV
years = np.arange(1, t_lifespan + 1)
discount_factors = (1 + r_discount) ** (-years)

# NPV dos Custos Totais (€)
npv_costs = total_capex + np.sum(total_opex * discount_factors)

# 4. Geração de Energia Anual com Degradação (MWh)
E_solar_t = solar_energy_annual_MWh * ((1 - solar_d) ** (years - 1))
E_wind_t = wind_energy_annual_MWh * ((1 - wind_d) ** (years - 1))
E_hydro_t = hydro_energy_annual_MWh * ((1 - hydro_d) ** (years - 1))
E_thermal_t = E_thermal_annual_MWh * ((1 - thermal_d) ** (years - 1))

E_total_t = E_solar_t + E_wind_t + E_hydro_t + E_thermal_t

# NPV da Energia Gerada (MWh)
npv_energy = np.sum(E_total_t * discount_factors)

# 5. Cálculo Final do LCOE (€/MWh)
lcoeA = npv_costs / npv_energy

print(f"Total CAPEX: {total_capex:,.2f} €")
print(f"OPEX Anual Total: {total_opex:,.2f} €/ano")
print(f"NPV dos Custos: {npv_costs:,.2f} €")
print(f"NPV da Energia: {npv_energy:,.2f} MWh")
print(f"LCOE: {lcoeA:.2f} €/MWh")

Total CAPEX: 33,272,000.00 €
OPEX Anual Total: 6,325,101.63 €/ano
NPV dos Custos: 102,965,178.18 €
NPV da Energia: 857,164.22 MWh
LCOE: 120.12 €/MWh


In [8]:
P_solar_B = 5
P_hydro_B = 0.84
P_wind_B = 4.7
P_thermal_B = 11.5

E_solar_B = 10153.99
E_hydro_B = 4577.55
E_wind_B = 18660.29
E_thermal_B = 38901.19


capex_vre_hydro_B = (P_solar_B * capex_solar_per_MW + 
                   P_wind_B * capex_wind_per_MW + 
                   P_hydro_B * capex_hydro_per_MW)

capex_thermal_B = P_thermal_B * capex_thermal_per_MW

first_year_opex_B = (P_solar_B * opex_solar_per_MW +
                   P_wind_B * opex_wind_per_MW +
                   P_hydro_B * opex_hydro_per_MW +
                   P_thermal_B * opex_thermal_fixed_per_MW)


total_capex = capex_vre_hydro_B + capex_thermal_B
total_fixed_opex = first_year_opex_B
variable_opex = E_thermal_annual_MWh * (fuel_euro_per_Wh + carbon_euro_per_Wh) * 1e6
total_opex = total_fixed_opex + variable_opex




# Vector de anos (1 a 20) e fatores de desconto NPV
years = np.arange(1, t_lifespan + 1)
discount_factors = (1 + r_discount) ** (-years)

# NPV dos Custos Totais (€)
npv_costs = total_capex + np.sum(total_opex * discount_factors)

# 4. Geração de Energia Anual com Degradação (MWh)
E_solar_t = solar_energy_annual_MWh * ((1 - solar_d) ** (years - 1))
E_wind_t = wind_energy_annual_MWh * ((1 - wind_d) ** (years - 1))
E_hydro_t = hydro_energy_annual_MWh * ((1 - hydro_d) ** (years - 1))
E_thermal_t = E_thermal_annual_MWh * ((1 - thermal_d) ** (years - 1))

E_total_t = E_solar_t + E_wind_t + E_hydro_t + E_thermal_t

# NPV da Energia Gerada (MWh)
npv_energy = np.sum(E_total_t * discount_factors)

# 5. Cálculo Final do LCOE (€/MWh)
lcoeB = npv_costs / npv_energy

print(f"Total CAPEX: {total_capex:,.2f} €")
print(f"OPEX Anual Total: {total_opex:,.2f} €/ano")
print(f"NPV dos Custos: {npv_costs:,.2f} €")
print(f"NPV da Energia: {npv_energy:,.2f} MWh")
print(f"LCOE: {lcoeB:.2f} €/MWh")

Total CAPEX: 27,316,000.00 €
OPEX Anual Total: 6,284,561.63 €/ano
NPV dos Custos: 96,562,487.89 €
NPV da Energia: 857,164.22 MWh
LCOE: 112.65 €/MWh


In [9]:
dlcoe = lcoeA-lcoeB

print(f"Delta LCOE (A-B): {dlcoe:.2f} €/MWh")

Delta LCOE (A-B): 7.47 €/MWh
